# District Heating MILP Formulation

This notebook is a clean implementation of the optimization problem described in `docs/optimization/district_heating_optimization_formulation.md`.

Purpose:
- keep the formulation compact and readable
- define a simple scenario in code
- build the MILP in Pyomo
- solve it with HiGHS
- extract results in a structure that can be reused later


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120
pd.options.display.float_format = '{:,.2f}'.format


## 1. Parameters

The parameters below directly follow the markdown formulation.


In [ ]:
@dataclass
class ModelParams:
    dt: float = 0.25
    n_hours: int = 24
    gas_price: float = 35.0
    co2_factor: float = 0.201
    co2_price: float = 60.0
    sto_capacity: float = 200.0
    sto_charge_max: float = 15.0
    sto_discharge_max: float = 15.0
    sto_loss: float = 0.000125
    sto_soc_init: float = 200.0
    hp_p_el_min: float = 1.0
    hp_p_el_max: float = 8.0
    hp_cop: float = 3.5
    boiler_q_min: float = 2.0
    boiler_q_max: float = 12.0
    boiler_eff: float = 0.97
    boiler_min_up: int = 4
    boiler_min_down: int = 4
    chp_p_el_min: float = 2.0
    chp_p_el_max: float = 6.0
    chp_eta_el: float = 0.40
    chp_eta_th: float = 0.48
    chp_min_up: int = 8
    chp_min_down: int = 8
    chp_startup_cost: float = 600.0
    soc_target: float = 160.0
    lambda_term: float = 25.0
    use_hard_soc_min: bool = False
    soc_min: float = 120.0
    hp_on_0: int = 0
    boiler_on_0: int = 0
    chp_on_0: int = 0

    @property
    def fuel_cost(self):
        return self.gas_price + self.co2_factor * self.co2_price

    @property
    def n_steps(self):
        return self.n_hours * 4

    @property
    def chp_th_per_el(self):
        return self.chp_eta_th / self.chp_eta_el


params = ModelParams()
params


## 2. Scenario Inputs

This notebook loads one 24-hour scenario directly from the repository data in `notebooks/optimization/Data`.
Demand and prices are hourly and then expanded to 15-minute steps for the MILP.


In [ ]:
def resolve_data_dir():
    cwd = Path.cwd()
    candidates = [
        cwd / 'Data',
        cwd / 'notebooks' / 'optimization' / 'Data',
    ]
    for base in [cwd] + list(cwd.parents):
        candidates.append(base / 'notebooks' / 'optimization' / 'Data')
        candidates.append(base / 'Data')

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError('Could not find notebooks/optimization/Data')


DATA_DIR = resolve_data_dir()
SCENARIO_START = pd.Timestamp('2024-03-01 00:00:00', tz='Europe/Berlin')

demand_raw = pd.read_csv(DATA_DIR / 'RawData_MeasuredHeadDemand.csv')
demand_ts = pd.to_datetime(demand_raw['Time Point'], utc=True).dt.tz_convert('Europe/Berlin')
demand_series = pd.Series(
    demand_raw['Measured Heat Demand[W]'].astype(float).to_numpy() / 1e6,
    index=demand_ts,
    name='demand_th',
).sort_index()
demand_hourly = demand_series.loc[SCENARIO_START:SCENARIO_START + pd.Timedelta(hours=params.n_hours - 1)].to_numpy()

price_raw = pd.read_csv(
    DATA_DIR / 'Gro_handelspreise_202403010000_202603020000_Stunde.csv',
    sep=';',
    decimal=',',
    low_memory=False,
)
price_ts = pd.to_datetime(price_raw['Datum von'], format='%d.%m.%Y %H:%M').dt.tz_localize('Europe/Berlin', ambiguous='infer', nonexistent='shift_forward')
price_series = pd.Series(
    pd.to_numeric(price_raw['Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen'], errors='coerce').to_numpy(),
    index=price_ts,
    name='price_el',
).sort_index()
price_hourly = price_series.loc[SCENARIO_START:SCENARIO_START + pd.Timedelta(hours=params.n_hours - 1)].to_numpy()

if len(demand_hourly) != params.n_hours or len(price_hourly) != params.n_hours:
    raise ValueError('Selected scenario window does not contain 24 hourly values for both demand and price')

demand_th = np.repeat(demand_hourly, 4)
price_el = np.repeat(price_hourly, 4)

scenario = pd.DataFrame({
    'demand_th': demand_hourly,
    'price_el': price_hourly,
}, index=pd.date_range(SCENARIO_START, periods=params.n_hours, freq='1h'))
scenario.index.name = 'timestamp_berlin'
scenario


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(scenario.index, scenario['demand_th'], marker='o')
axes[0].set_ylabel('MW_th')
axes[0].set_title('Hourly heat demand from repo data')
axes[1].plot(scenario.index, scenario['price_el'], marker='o', color='tab:red')
axes[1].set_ylabel('EUR/MWh_el')
axes[1].set_title('Hourly electricity price from repo data')
axes[1].set_xlabel('Timestamp (Berlin)')
plt.tight_layout()


## 3. MILP Builder

The model below follows the variable names and constraints from the formulation markdown.


In [ ]:
def add_min_up_down_constraints(model, on_var, start_var, stop_var, min_up, min_down, prefix):
    # Minimum up-time: once started, the unit must remain on for a fixed number of steps.
    if min_up > 0:
        def min_up_rule(m, t):
            start_idx = max(0, t - min_up + 1)
            return sum(start_var[i] for i in range(start_idx, t + 1)) <= on_var[t]
        model.add_component(f'{prefix}_min_up', pyo.Constraint(model.T, rule=min_up_rule))

    # Minimum down-time: once stopped, the unit must remain off for a fixed number of steps.
    if min_down > 0:
        def min_down_rule(m, t):
            start_idx = max(0, t - min_down + 1)
            return sum(stop_var[i] for i in range(start_idx, t + 1)) <= 1 - on_var[t]
        model.add_component(f'{prefix}_min_down', pyo.Constraint(model.T, rule=min_down_rule))


def build_model(params, demand_th, price_el):
    if len(demand_th) != params.n_steps or len(price_el) != params.n_steps:
        raise ValueError('Inputs must have length params.n_steps')

    m = pyo.ConcreteModel('DistrictHeatingMILP')
    m.T = pyo.RangeSet(0, params.n_steps - 1)
    m.demand_th = pyo.Param(m.T, initialize={t: float(demand_th[t]) for t in range(params.n_steps)})
    m.price_el = pyo.Param(m.T, initialize={t: float(price_el[t]) for t in range(params.n_steps)})

    # Binary commitment variables.
    m.hp_on = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_on = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_on = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_start = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_stop = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_start = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_stop = pyo.Var(m.T, domain=pyo.Binary)
    m.sto_mode_charge = pyo.Var(m.T, domain=pyo.Binary)

    # Continuous dispatch and storage variables.
    m.hp_el_in = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.hp_th_out = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.boiler_th_out = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.boiler_fuel_in = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_el_out = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_th_out = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_fuel_in = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_charge = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_discharge = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_soc = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_shortfall = pyo.Var(domain=pyo.NonNegativeReals)

    # Heat balance: total supplied heat minus storage charging must match demand each step.
    m.heat_balance = pyo.Constraint(
        m.T,
        rule=lambda mm, t: mm.hp_th_out[t] + mm.boiler_th_out[t] + mm.chp_th_out[t] + mm.sto_th_discharge[t] - mm.sto_th_charge[t] == mm.demand_th[t],
    )

    # Storage capacity bound.
    m.sto_soc_bound = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_soc[t] <= params.sto_capacity)

    # Storage mode logic: only charge when in charge mode.
    m.sto_charge_limit = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_charge[t] <= params.sto_charge_max * mm.sto_mode_charge[t])

    # Storage mode logic: only discharge when not in charge mode.
    m.sto_discharge_limit = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_discharge[t] <= params.sto_discharge_max * (1 - mm.sto_mode_charge[t]))

    # Storage dynamics: state of charge evolves with charge, discharge, and standing loss.
    def soc_rule(mm, t):
        prev_soc = params.sto_soc_init if t == 0 else mm.sto_th_soc[t - 1]
        return mm.sto_th_soc[t] == prev_soc + params.dt * mm.sto_th_charge[t] - params.dt * mm.sto_th_discharge[t] - params.sto_loss
    m.sto_soc_dynamics = pyo.Constraint(m.T, rule=soc_rule)

    # Soft terminal condition: penalize ending below the target storage level.
    m.sto_terminal_shortfall = pyo.Constraint(expr=m.sto_th_shortfall >= params.soc_target - m.sto_th_soc[params.n_steps - 1])

    # Optional hard terminal condition: enforce a minimum end-of-horizon storage level.
    if params.use_hard_soc_min:
        m.sto_terminal_min = pyo.Constraint(expr=m.sto_th_soc[params.n_steps - 1] >= params.soc_min)

    # Heat pump operating window when committed.
    m.hp_min = pyo.Constraint(m.T, rule=lambda mm, t: params.hp_p_el_min * mm.hp_on[t] <= mm.hp_el_in[t])
    m.hp_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_el_in[t] <= params.hp_p_el_max * mm.hp_on[t])

    # Heat pump conversion: thermal output equals COP times electrical input.
    m.hp_cop = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_th_out[t] == params.hp_cop * mm.hp_el_in[t])

    # Boiler operating window when committed.
    m.boiler_min = pyo.Constraint(m.T, rule=lambda mm, t: params.boiler_q_min * mm.boiler_on[t] <= mm.boiler_th_out[t])
    m.boiler_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_th_out[t] <= params.boiler_q_max * mm.boiler_on[t])

    # Boiler fuel conversion: thermal output is linked to fuel input by efficiency.
    m.boiler_eta = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_fuel_in[t] * params.boiler_eff == mm.boiler_th_out[t])

    # Boiler startup indicator: becomes active when the boiler switches on.
    m.boiler_start_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_start[t] >= mm.boiler_on[t] - (params.boiler_on_0 if t == 0 else mm.boiler_on[t - 1]))

    # Boiler shutdown indicator: becomes active when the boiler switches off.
    m.boiler_stop_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_stop[t] >= (params.boiler_on_0 if t == 0 else mm.boiler_on[t - 1]) - mm.boiler_on[t])
    add_min_up_down_constraints(m, m.boiler_on, m.boiler_start, m.boiler_stop, params.boiler_min_up, params.boiler_min_down, 'boiler')

    # CHP operating window when committed.
    m.chp_min = pyo.Constraint(m.T, rule=lambda mm, t: params.chp_p_el_min * mm.chp_on[t] <= mm.chp_el_out[t])
    m.chp_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_el_out[t] <= params.chp_p_el_max * mm.chp_on[t])

    # CHP heat-power coupling: thermal output is proportional to electrical output.
    m.chp_heat = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_th_out[t] == params.chp_th_per_el * mm.chp_el_out[t])

    # CHP fuel conversion: electrical output is linked to fuel input by electrical efficiency.
    m.chp_eta = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_fuel_in[t] * params.chp_eta_el == mm.chp_el_out[t])

    # CHP startup indicator: becomes active when the CHP switches on.
    m.chp_start_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_start[t] >= mm.chp_on[t] - (params.chp_on_0 if t == 0 else mm.chp_on[t - 1]))

    # CHP shutdown indicator: becomes active when the CHP switches off.
    m.chp_stop_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_stop[t] >= (params.chp_on_0 if t == 0 else mm.chp_on[t - 1]) - mm.chp_on[t])
    add_min_up_down_constraints(m, m.chp_on, m.chp_start, m.chp_stop, params.chp_min_up, params.chp_min_down, 'chp')

    # Objective: operating cost plus startup cost minus CHP power market revenue plus terminal storage penalty.
    m.objective = pyo.Objective(
        expr=sum(
            m.hp_el_in[t] * m.price_el[t] * params.dt
            + m.boiler_fuel_in[t] * params.fuel_cost * params.dt
            + m.chp_fuel_in[t] * params.fuel_cost * params.dt
            + params.chp_startup_cost * m.chp_start[t]
            - m.chp_el_out[t] * m.price_el[t] * params.dt
            for t in m.T
        ) + params.lambda_term * m.sto_th_shortfall,
        sense=pyo.minimize,
    )

    return m


## 4. Solve


In [ ]:
model = build_model(params, demand_th, price_el)
solver = pyo.SolverFactory('appsi_highs')
result = solver.solve(model)
print('Termination condition:', result.solver.termination_condition)


## 5. Extract Results

The result table is intentionally flat and explicit so it can be reused in later notebooks or exported.


In [ ]:
dispatch = pd.DataFrame({
    'demand_th': demand_th,
    'price_el': price_el,
    'hp_on': [pyo.value(model.hp_on[t]) for t in model.T],
    'boiler_on': [pyo.value(model.boiler_on[t]) for t in model.T],
    'chp_on': [pyo.value(model.chp_on[t]) for t in model.T],
    'hp_el_in': [pyo.value(model.hp_el_in[t]) for t in model.T],
    'hp_th_out': [pyo.value(model.hp_th_out[t]) for t in model.T],
    'boiler_th_out': [pyo.value(model.boiler_th_out[t]) for t in model.T],
    'boiler_fuel_in': [pyo.value(model.boiler_fuel_in[t]) for t in model.T],
    'chp_el_out': [pyo.value(model.chp_el_out[t]) for t in model.T],
    'chp_th_out': [pyo.value(model.chp_th_out[t]) for t in model.T],
    'chp_fuel_in': [pyo.value(model.chp_fuel_in[t]) for t in model.T],
    'sto_th_charge': [pyo.value(model.sto_th_charge[t]) for t in model.T],
    'sto_th_discharge': [pyo.value(model.sto_th_discharge[t]) for t in model.T],
    'sto_th_soc': [pyo.value(model.sto_th_soc[t]) for t in model.T],
}, index=pd.Index(range(params.n_steps), name='step'))

summary = pd.Series({
    'objective_eur': pyo.value(model.objective),
    'terminal_soc_mwh': pyo.value(model.sto_th_soc[params.n_steps - 1]),
    'storage_shortfall_mwh': pyo.value(model.sto_th_shortfall),
    'hp_heat_mwh': dispatch['hp_th_out'].sum() * params.dt,
    'boiler_heat_mwh': dispatch['boiler_th_out'].sum() * params.dt,
    'chp_heat_mwh': dispatch['chp_th_out'].sum() * params.dt,
    'chp_power_mwh': dispatch['chp_el_out'].sum() * params.dt,
})

summary.to_frame('value')


In [ ]:
dispatch.head(12)


## 6. Visual Check


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
x = dispatch.index

axes[0].plot(x, dispatch['demand_th'], color='black', linewidth=2, label='Demand')
axes[0].stackplot(
    x,
    dispatch['hp_th_out'],
    dispatch['boiler_th_out'],
    dispatch['chp_th_out'],
    dispatch['sto_th_discharge'],
    labels=['HP heat', 'Boiler heat', 'CHP heat', 'Storage discharge'],
    alpha=0.85,
)
axes[0].plot(x, dispatch['sto_th_charge'], linestyle='--', color='tab:blue', label='Storage charge')
axes[0].set_ylabel('MW_th')
axes[0].legend(loc='upper right', ncol=3)

axes[1].plot(x, dispatch['price_el'], color='tab:red', label='Electricity price')
axes[1].plot(x, dispatch['hp_el_in'], color='tab:purple', label='HP electricity in')
axes[1].plot(x, dispatch['chp_el_out'], color='tab:green', label='CHP electricity out')
axes[1].set_ylabel('MW / EUR per MWh')
axes[1].legend(loc='upper right')

axes[2].plot(x, dispatch['sto_th_soc'], color='tab:orange', linewidth=2, label='Storage SoC')
axes[2].axhline(params.soc_target, color='grey', linestyle='--', label='SoC target')
if params.use_hard_soc_min:
    axes[2].axhline(params.soc_min, color='red', linestyle=':', label='SoC minimum')
axes[2].set_ylabel('MWh_th')
axes[2].set_xlabel('15-minute step')
axes[2].legend(loc='upper right')

plt.tight_layout()
